# 3D example
This notebook shows a simple analysis pipeline for running and analyzing a 3D simulation with DISCO-DJ.

I### Imports

In [ ]:
# Import modules
import os
from matplotlib import pyplot as plt
%matplotlib inline
import seaborn as sns
sns.set_style("ticks")
import jax
from jax import config
import numpy as np
os.environ["USE_MASSIVE_NEUTRINOS"] = "1" #Def we use normal DISCO-DJ or Massive neutrinos version
from discodj import DiscoDJ
plt.rcParams['image.cmap'] = "rocket"
print(jax.__version__)

In [ ]:
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
# os.environ["XLA_PYTHON_CLIENT_ALLOCATOR"] = "platform"
#os.environ["XLA_PYTHON_CLIENT_MEM_FRACTION"] = "0.95" # maximize available memory (might need to tune this in case you're running out of memory)

### Simulation settings
DISCO-DJ can be run on a GPU (*much* faster) or on a CPU. The following cell detects if a GPU is available and sets the device accordingly.

In [ ]:
# Detect if a GPU is available
devices = jax.devices()
device = "gpu" if np.any([d.platform == "gpu" for d in devices]) else "cpu"
print(device)

Now, we set the name of the analysis and the spatial dimension. The precision can be set to "single" (float32) or "double" (float64; in that case, the Jax config needs to be updated). DISCO-DJ is written in an class-based (yet functional) way, so you need to define a DiscoDJ object that will be used to perform all computations.

In [ ]:
# Set the parameters
name = "3D_analysis"
dim = 3

precision = "single"
if precision == "double":
    config.update("jax_enable_x64", True)

# The cosmology can be provided as a dictionary or a string for a pre-defined cosmology (e.g. "CamelsCV" for the Camels Cosmic Variance suite).
# cosmo = "CamelsCV"
cosmo = dict(Omega_c=0.259605,  # cold dark matter content
             Omega_b=0.0488911,  # baryonic content (note: Disco-DJ so far only performs N-body simulations, no hydro; this is only used for the linear power spectrum!)
             h=0.67742,  # dimensionless Hubble constant
             n_s=0.96822,  # scalar spectral index
             sigma8=0.808992  # amplitude of matter density fluctuations at a scale of 8 Mpc/h
             )


# Define the boxsize and resolution
boxsize = 300.0  # in Mpc/h
res = 128  # the particles live on a Lagrangrian grid of resolution (res)^dim
# Define DISCO-DJ object
dj = DiscoDJ(dim=dim, res=res, name=name, device=device, precision=precision, boxsize=boxsize, cosmo=cosmo)
print(dj)

Next, we compute the timetable for the cosmological growth functions.

In [ ]:
# Compute the linear power spectrum
dj = dj.with_timetables()
dj = dj.with_linear_ps(transfer_function="Eisenstein-Hu")

### Initial conditions

In [ ]:
white_noise_field = dj.get_ngenic_noise(seed=24680)  # a white noise field, defined in real space
#plt.figure(figsize=(6, 6))
#plt.imshow(white_noise_field[0, :, :], cmap="RdBu")

k_noise, Pk_noise, _ = dj.evaluate_power_spectrum(white_noise_field, compute_std=False, bins=1000)
#plt.figure(figsize=(6, 6))
#plt.loglog(k_noise, Pk_noise)
#plt.xlim(5 * np.nanmin(k_noise), 1.1 * dj.k_nyquist)
#plt.ylim([1e-1, 2 * np.nanmax(Pk_noise)])
#plt.xlabel(r"$k$ [$h$/Mpc]")
#plt.ylabel(r"$P(k)$ [$h^{-3}$Mpc$^3$]")

#We can now plot a slice of the initial conditions:

dj = dj.with_ics(white_noise_space="real", white_noise_field=white_noise_field, convert_to_numpy=True)
#plt.figure(figsize=(6, 6))
#plt.imshow(dj.delta_ini[0, :, :], cmap="RdBu")

#We can also compute the power spectrum of the pre-initial conditions:

k_pre, Pk_pre, _ = dj.evaluate_power_spectrum(dj.delta_ini, compute_std=False, bins=80)
#plt.figure(figsize=(6, 6))
#plt.loglog(k_pre, Pk_pre)
#plt.xlabel(r"$k$ [$h$/Mpc]")
#plt.ylabel(r"$P(k)$ [$h^{-3}$Mpc$^3$]")

### Lagrangian perturbation theory (LPT)

In [ ]:
n_order = 2
dj = dj.with_lpt(n_order=n_order, convert_to_numpy=True)  # convert_to_numpy as above; note that this breaks differentiability

### Particle-mesh (PM) N-body simulation


In [ ]:
# These options should be alright in many scenarios
stepper = "bullfrog"
method = "pm"
res_pm = 2 * dj.res
time_var = "D"
antialias = 0
grad_kernel_order = 4
laplace_kernel_order = 0
worder = 2
n_resample = 1
deconvolve = False
nlpt_order_ics = n_order
chunk_size = None  # if you are running out of GPU memory, doing the mass assignment in smaller chunks might help (e.g. chunk_size = dj.res ** dj.dim // 16)
numsteps = 10  # number of steps to be performed
a_ini = 0.02  # initial scale factor of the simulation (where it is initialized with LPT)
a_end = 1.0  # final scale factor of the simulation
a_list = np.linspace(a_ini, a_end, numsteps, endpoint=False)

delta_slice = np.zeros((numsteps, 8*dj.res, 8*dj.res))

In [ ]:
X_sim, P_sim, a_sim = dj.run_nbody(a_ini=a_ini, a_end=a_end, n_steps=numsteps, res_pm=res_pm,
                                   time_var=time_var, stepper=stepper, method=method, antialias=antialias,
                                   grad_kernel_order=grad_kernel_order, laplace_kernel_order=laplace_kernel_order,
                                   nlpt_order_ics=nlpt_order_ics, n_resample=n_resample,
                                   deconvolve=deconvolve, return_displacement=False, chunk_size=chunk_size,
                                   convert_to_numpy=True)  # convert_to_numpy: same as above for the ICs and LPT


In [ ]:
#for i, a in enumerate(a_list):
#    X_sim, P_sim, a_sim = dj.run_nbody(a_ini=a, a_end=a_list[i + 1] if i + 1 < len(a_list) else a_end, n_steps=1, res_pm=res_pm,
#                                    time_var=time_var, stepper=stepper, method=method, antialias=antialias,
#                                    grad_kernel_order=grad_kernel_order, laplace_kernel_order=laplace_kernel_order,
#                                    nlpt_order_ics=nlpt_order_ics, n_resample=n_resample,
#                                    deconvolve=deconvolve, return_displacement=False, chunk_size=chunk_size,
#                                    convert_to_numpy=True)  # convert_to_numpy: same as above for the ICs and LPT
#    dj = dj.with_external_ics(pos=X_sim.reshape(-1, 3), vel=P_sim.reshape(-1, 3))
#    delta_slice[i] = dj.compute_field_quantity_from_particles(pos=X_sim.reshape(-1, 3), res=8 * dj.res, n_resample=16, worder=4, fixed_inds=[0, None, None])

### Analysis

Let's compute the power spectrum:

In [ ]:
delta_sim = dj.get_delta_from_pos(X_sim, res=2 * dj.res, n_resample=8)

In [ ]:
k, Pk, _ = dj.evaluate_power_spectrum(delta_sim, bins=dj.res//3, deconvolve=True, worder=2)
Pk_linear = dj.evaluate_linear_ps(a_end, k)  # linear expectation
_, Pk_linear_realized, _ = dj.evaluate_power_spectrum(dj.delta_ini, bins=dj.res//3)  # linear realization

In [ ]:
plt.figure(figsize=(10, 6))
plt.loglog(k, Pk, "k-", label="Non-linear")
plt.loglog(k, Pk_linear, "b:", label="Linear (Expectation)")
plt.loglog(k, Pk_linear_realized, "r-.", label="Linear (Realized)")
plt.xlabel(r"$k$ [$h$/Mpc]")
plt.ylabel(r"$P(k)$")
plt.legend(fontsize=18)

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(16, 8))
axs[0].imshow(np.log10(1.01 + delta_sim.mean(0))); axs[0].set_title("Avg.")
axs[1].imshow(np.log10(1.01 + delta_sim[0, :, :])); axs[1].set_title("Slice")

#fig, axs = plt.subplots(2, numsteps//2, figsize=(32, 10))
#axs = axs.flatten()
#
#for i, a in enumerate(a_list):
#    axs[i].imshow(np.log10(1.001 + delta_slice[i]))
#    axs[i].set_title(f"a = {a:.2f}")